# เทรนโมเดลเสียงของคุณ (RVC) บน GPU ฟรี

ใช้ได้ทั้ง **Google Colab** (Runtime → Change runtime type → T4 GPU) และ **Kaggle** (Settings → Accelerator → GPU T4)

1. ในสตูดิโอพากเสียง กด **ดาวน์โหลดเทคที่เก็บไว้ (.zip)** แล้วอัปโหลด `my-voice-dataset.zip` เข้ามาที่นี่
2. รันทีละช่องจากบนลงล่าง (เทรนรอบแรกประมาณ 30–60 นาที)
3. ดาวน์โหลด `.pth` และ `.index` ที่ได้ แล้วอัปโหลดกลับที่ช่อง **สอนเสียงของคุณ**

ใช้เฉพาะเสียงของคุณเอง หรือเสียงที่เจ้าของอนุญาตแล้วเท่านั้น

> หมายเหตุ: โน้ตบุ๊กนี้ใช้ [Applio](https://github.com/IAHispano/Applio) เวอร์ชันที่ระบุไว้ ถ้าคำสั่งไหนแจ้งว่าไม่รู้จักตัวเลือก ให้รัน `python core.py <คำสั่ง> -h` เพื่อดูชื่อตัวเลือกของเวอร์ชันนั้น

In [ ]:
# 1) ตั้งค่า
MODEL_NAME = "my-voice-v1"   # เปลี่ยนเป็น v2, v3 ... ทุกครั้งที่เทรนใหม่
EPOCHS = 200                 # เสียง 10–30 นาที ใช้ 150–300 กำลังดี
BATCH_SIZE = 8
SAMPLE_RATE = 40000
APPLIO_VERSION = "3.7.2"


In [ ]:
# 2) ติดตั้ง Applio (ประมาณ 3–5 นาที)
!git clone --depth 1 --branch {APPLIO_VERSION} https://github.com/IAHispano/Applio.git /content/Applio
%cd /content/Applio
!pip install -q -r requirements-engine.txt
!python core.py prerequisites --models --no-exe


In [ ]:
# 3) แตกไฟล์เสียงที่อัปโหลด (Colab: อัปโหลดผ่านแถบไฟล์ด้านซ้าย / Kaggle: Add data)
import glob, zipfile, os
zips = glob.glob("/content/*.zip") + glob.glob("/kaggle/input/**/*.zip", recursive=True)
assert zips, "ไม่พบไฟล์ .zip กรุณาอัปโหลด my-voice-dataset.zip ก่อน"
DATASET = "/content/dataset"
os.makedirs(DATASET, exist_ok=True)
zipfile.ZipFile(zips[0]).extractall(DATASET)
wavs = glob.glob(f"{DATASET}/*.wav")
print(f"พบ {len(wavs)} ไฟล์เสียง")


In [ ]:
# 4) เตรียมข้อมูล → ดึงลักษณะเสียง → เทรน → สร้าง index
!python core.py preprocess --model-name {MODEL_NAME} --dataset-path {DATASET} --sample-rate {SAMPLE_RATE}
!python core.py extract --model-name {MODEL_NAME} --f0-method rmvpe --sample-rate {SAMPLE_RATE} --gpu 0
!python core.py train --model-name {MODEL_NAME} --sample-rate {SAMPLE_RATE} --total-epoch {EPOCHS} --batch-size {BATCH_SIZE} --save-every-epoch 50 --gpu 0
!python core.py index --model-name {MODEL_NAME}


In [ ]:
# 5) รวมไฟล์ที่ต้องดาวน์โหลด
import shutil, glob
logs = f"/content/Applio/logs/{MODEL_NAME}"
# ไฟล์ G_*.pth / D_*.pth เป็นจุดพักการเทรน ไม่ใช่โมเดลสำหรับใช้งาน
pth = sorted([p for p in glob.glob(f"{logs}/{MODEL_NAME}_*e_*s.pth")], key=os.path.getmtime)[-1]
idx = glob.glob(f"{logs}/*.index")
out = f"/content/{MODEL_NAME}"
os.makedirs(out, exist_ok=True)
shutil.copy(pth, f"{out}/{MODEL_NAME}.pth")
if idx: shutil.copy(idx[0], f"{out}/{MODEL_NAME}.index")
shutil.make_archive(out, "zip", out)
print("ดาวน์โหลด", out + ".zip", "แล้วอัปโหลด .pth และ .index เข้าแอป")
